In [ ]:
import os
import nibabel as nib
import numpy as np

# === Dossiers ===
images_dir = "/home/amenacer/Stage/base_de_donnees/3D/3D-short-gz"
masks_dir = "/home/amenacer/Stage/base_de_donnees/segmentation"
output_dir = "/home/amenacer/Stage/base_de_donnees/masked"

os.makedirs(output_dir, exist_ok=True)

# === Parcourir les masques
for mask_file in os.listdir(masks_dir):
    if mask_file.endswith(".nii.gz"):
        mask_path = os.path.join(masks_dir, mask_file)

        # Reconstituer le nom de l’image correspondante
        base_name = mask_file.replace(".nii.gz", "")
        image_file = base_name + "_0000.nii.gz"
        image_path = os.path.join(images_dir, image_file)

        if os.path.exists(image_path):
            # Charger image et masque
            image_nii = nib.load(image_path)
            mask_nii = nib.load(mask_path)

            image_data = image_nii.get_fdata()
            mask_data = mask_nii.get_fdata()

            # Appliquer le masque
            masked_data = image_data * mask_data

            # Sauvegarde
            masked_nii = nib.Nifti1Image(masked_data, affine=image_nii.affine)
            masked_output_path = os.path.join(output_dir, mask_file)  # même nom que le masque
            nib.save(masked_nii, masked_output_path)

            print(f"✅ Masqué : {image_file} + {mask_file} → {masked_output_path}")
        else:
            print(f"❌ Image non trouvée pour : {mask_file} (cherché : {image_file})")


In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd

masked_dir = "/home/amenacer/Stage/base_de_donnees/masked"
output_csv = "/home/amenacer/Stage/base_de_donnees/descripteurs.csv"

results = []

for file in os.listdir(masked_dir):
    if file.endswith(".nii.gz"):
        path = os.path.join(masked_dir, file)
        img = nib.load(path)
        data = img.get_fdata()

        # S'assurer que les données sont 3D (X, Y, T)
        if data.ndim != 3:
            print(f"❌ Skipped {file}: data is not 3D.")
            continue

        # Calcul de l'aire à chaque frame (somme des voxels non-nuls)
        areas = [np.count_nonzero(data[:, :, t]) for t in range(data.shape[2])]
        areas = np.array(areas)

        # Calcul des descripteurs
        min_area = areas.min()
        max_area = areas.max()
        mean_area = areas.mean()
        std_area = areas.std()

        # Pente montante = max de la dérivée positive
        diffs = np.diff(areas)
        ascending_slope = np.max(diffs)
        descending_slope = np.min(diffs)

        results.append({
            "fichier": file,
            "min_area": min_area,
            "max_area": max_area,
            "mean_area": mean_area,
            "std_area": std_area,
            "ascending_slope": ascending_slope,
            "descending_slope": descending_slope
        })

        print(f"✅ {file} traité → min: {min_area}, max: {max_area}")

# Export CSV
df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)
print(f"\n📄 Fichier descripteurs sauvegardé : {output_csv}")


In [ ]:
import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

masked_dir = "/home/amenacer/Stage/base_de_donnees/masked"
output_plot_dir = "/home/amenacer/Stage/base_de_donnees/plots"
os.makedirs(output_plot_dir, exist_ok=True)

def detect_anomaly(area_curve):
    diffs = np.diff(area_curve)
    if np.all(diffs == 0):
        return "Plateau (aucune variation)"
    if np.max(np.abs(diffs)) > 0.5 * np.max(area_curve):
        return "Saut brutal"
    if np.std(diffs) < 10:
        return "Courbe très stable"
    return None

for file in os.listdir(masked_dir):
    if file.endswith(".nii.gz"):
        path = os.path.join(masked_dir, file)
        img = nib.load(path)
        data = img.get_fdata()

        if data.ndim != 3:
            continue

        areas = [np.count_nonzero(data[:, :, t]) for t in range(data.shape[2])]
        anomaly = detect_anomaly(np.array(areas))

        # Tracer la courbe
        plt.figure(figsize=(8, 4))
        plt.plot(areas, marker='o')
        plt.title(f"Aire vs Temps : {file}")
        plt.xlabel("Temps (frame)")
        plt.ylabel("Aire segmentée")
        if anomaly:
            plt.suptitle(f"⚠️ Anomalie détectée : {anomaly}", color="red", fontsize=10)
        plt.grid(True)
        plt.tight_layout()

        save_path = os.path.join(output_plot_dir, file.replace(".nii.gz", ".png"))
        plt.savefig(save_path)
        plt.close()

        print(f"📈 Courbe générée pour {file} {'❗' + anomaly if anomaly else '✅ OK'}")

print(f"\n🗂️ Courbes enregistrées dans : {output_plot_dir}")


In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd

masked_dir = "/home/amenacer/Stage/base_de_donnees/masked"
csv_output = "/home/amenacer/Stage/base_de_donnees/anomalies.csv"

results = []

def detect_anomaly(area_curve):
    diffs = np.diff(area_curve)
    if np.all(diffs == 0):
        return "plateau"
    if np.max(np.abs(diffs)) > 0.5 * np.max(area_curve):
        return "saut brutal"
    if np.std(area_curve) < 50:
        return "variation très faible"
    if np.std(diffs) > 3 * np.mean(np.abs(diffs)):
        return "instabilité"
    return "normal"

for file in os.listdir(masked_dir):
    if file.endswith(".nii.gz"):
        path = os.path.join(masked_dir, file)
        img = nib.load(path)
        data = img.get_fdata()

        if data.ndim != 3:
            continue

        areas = [np.count_nonzero(data[:, :, t]) for t in range(data.shape[2])]
        anomaly = detect_anomaly(np.array(areas))

        results.append({
            "fichier": file,
            "min_area": np.min(areas),
            "max_area": np.max(areas),
            "mean_area": np.mean(areas),
            "std_area": np.std(areas),
            "anomalie": anomaly
        })

        print(f"📊 {file} → {anomaly}")

# Enregistrement CSV
df = pd.DataFrame(results)
df.to_csv(csv_output, index=False)
print(f"\n📄 Fichier anomalies sauvegardé : {csv_output}")


In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

masked_dir = "/home/amenacer/Stage/base_de_donnees/masked"
output_plot_dir = "/home/amenacer/Stage/base_de_donnees/plots_anomalies"
output_csv = "/home/amenacer/Stage/base_de_donnees/anomalies.csv"

os.makedirs(output_plot_dir, exist_ok=True)

def detect_anomaly(area_curve):
    diffs = np.diff(area_curve)

    if np.all(diffs == 0):
        return "Plateau total"
    if np.max(np.abs(diffs)) > 0.5 * np.max(area_curve):
        return "Saut brutal"
    if np.std(area_curve) < 50:
        return "Très faible variation"
    if np.max(diffs) < 10:
        return "Pas de montée nette"
    return "normale"

results = []

for file in os.listdir(masked_dir):
    if file.endswith(".nii.gz"):
        path = os.path.join(masked_dir, file)
        img = nib.load(path)
        data = img.get_fdata()

        if data.ndim != 3:
            continue

        areas = np.array([np.count_nonzero(data[:, :, t]) for t in range(data.shape[2])])
        label = detect_anomaly(areas)

        # Enregistrement dans le CSV
        results.append({
            "fichier": file,
            "min_area": np.min(areas),
            "max_area": np.max(areas),
            "std_area": np.std(areas),
            "type": label
        })

        # Plot de la courbe
        plt.figure(figsize=(8, 4))
        color = 'red' if label != "normale" else 'green'
        plt.plot(areas, marker='o', color=color)
        plt.title(f"{file}", fontsize=10)
        plt.suptitle(f"Aire vs Temps – {label}", color=color, fontsize=12)
        plt.xlabel("Temps (frame)")
        plt.ylabel("Aire segmentée")
        plt.grid(True)
        plt.tight_layout()

        save_name = os.path.join(output_plot_dir, file.replace(".nii.gz", ".png"))
        plt.savefig(save_name)
        plt.close()

        print(f"📈 {file} → {label} → {save_name}")

# Export CSV
df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)
print(f"\n✅ Anomalies détectées et sauvegardées dans : {output_csv}")
print(f"🖼️ Figures enregistrées dans : {output_plot_dir}")
